# ДЗ-2. Flow matching и диффузия: от условного таргета к генерации

Курс «Дополнительные архитектуры генеративного ИИ», ФКН НИУ ВШЭ, программа ИПИИ.

Выдаётся 06.10, сдаётся 13.10. Вес 30 % итоговой оценки.
Трудоёмкость 8-10 часов. Все вычисления на CPU.

---

Сдаётся этот же ноутбук, заполненный, с сохранёнными результатами выполнения
ячеек, и файл `measurements.csv`.

## Направление времени

**На занятиях 4 и 5 ориентация разная, и это главный источник ошибок в этом
задании.** Здесь используется соглашение занятия 5.

- $\tau=0$ - чистые данные, $X_0\sim p_{\text{data}}$;
- $\tau=1$ - чистый шум, $X_1\sim\mathcal N(0,I)$;
- зашумление идёт от $0$ к $1$;
- **генерация идёт от $1$ к $0$.**

На занятии 4 в конструкции flow matching $x_0$ был источником, а $x_1$ целью,
то есть наоборот. Ни одна формула отсюда не переносится туда без проверки
направления.

## Нотация

Точки данных двумерные: $x\in\mathbb R^2$. Батч имеет форму `[B, 2]`, время -
форму `[B, 1]`: у каждого элемента батча своё $\tau$.

**Гауссово зашумление.**

$$
X_\tau=\alpha_\tau X_0+\sigma_\tau\epsilon,\qquad \epsilon\sim\mathcal N(0,I)
$$

**Нормированный угловой путь.** Расписание зафиксировано:

$$
\alpha_\tau=\cos\frac{\pi\tau}{2},\qquad \sigma_\tau=\sin\frac{\pi\tau}{2}
$$

Тогда $\alpha_\tau^2+\sigma_\tau^2=1$, при $\tau=0$ получается $X_0$, при
$\tau=1$ получается $\epsilon$. Функция `angular_schedule(tau)` дана готовой и
возвращает пару `(alpha, sigma)` формы `[B, 1]`.

**Три разных объекта, которые в литературе обозначают буквой $v$.**

| Объект | Что это | От чего зависит |
|---|---|---|
| $v_\tau^{\mathrm{VP}}=\alpha_\tau\epsilon-\sigma_\tau X_0$ | условный velocity-таргет | от скрытой пары $(X_0,\epsilon)$ |
| $U_\tau=\frac{\pi}{2}v_\tau^{\mathrm{VP}}$ | условная transport velocity | от той же пары |
| $v_\tau(x)=\mathbb E[U_\tau\mid X_\tau=x]$ | marginal vector field | только от $(x,\tau)$ |

Первые два случайны при фиксированном $x_\tau$: одна и та же точка получается из
разных пар $(X_0,\epsilon)$. Третий - однозначная функция, и именно он нужен
для генерации.

Множитель $\frac{\pi}{2}$ берётся из $\dot\phi_\tau$ при $\phi_\tau=\pi\tau/2$:

$$
\frac{dX_\tau}{d\tau}=\dot\alpha_\tau X_0+\dot\sigma_\tau\epsilon
=\frac{\pi}{2}\left(-\sigma_\tau X_0+\alpha_\tau\epsilon\right)
=\frac{\pi}{2}v_\tau^{\mathrm{VP}}
$$

**Обученная сеть.** Сеть предсказывает $v^{\mathrm{VP}}$, поэтому поле для
интегрирования получается умножением:

$$
v_\theta^{\mathrm{ODE}}(x,\tau)=\frac{\pi}{2}\hat v_\theta^{\mathrm{VP}}(x,\tau)
$$

Это `model_ode_field`, тоже дана готовой.

**Солвер и выборка.** Солвер - численная схема интегрирования; здесь это явный
Эйлер и данный готовым RK4. Выборка - запуск солвера от шума до данных. Дальше
слово «солвер» относится к схеме, а не к обёртке.

**Число вызовов сети.** $N_{\mathrm{FE}}$ - сколько раз вызвана сеть. У Эйлера
один шаг солвера - один вызов, у RK4 один шаг - четыре вызова. Арифметика с
тензорами, расписание и подсчёт метрики в $N_{\mathrm{FE}}$ не входят. Размер
батча на $N_{\mathrm{FE}}$ не влияет: один батчевый вызов обслуживает все
траектории сразу.

## Контракт трёх функций

Общее для всех трёх:

- все тензоры на CPU, обычные strided, не sparse;
- dtype одинаковый у всех входов, `torch.float32` или `torch.float64`; выход в
  том же dtype;
- входы конечны, `NaN` и бесконечностей нет;
- функции не меняют входы на месте;
- функции ничего не сэмплируют сами: весь случайный материал приходит
  аргументами;
- повторный вызов с теми же аргументами даёт тот же результат; для `vp_loss`
  это понимается при детерминированной `model`.

**Что возвращают `model` и `field`.** `model(x, tau)` и `field(x, tau)` дают
тензор формы `[B, 2]` того же dtype, что `x`, с конечными значениями. Опираться
на что-то сверх этого нельзя.

**Градиенты по входам в контракт не входят.** Требуется только, чтобы
производные доходили до параметров `model`. Производные по `x_0`, `eps` и `tau`
не проверяются, и отрывать их от графа не запрещено.

### `vp_target(x_0, eps, tau) -> Tensor`

Формы: `x_0` и `eps` - `[B, 2]`, `tau` - `[B, 1]`, выход - `[B, 2]`, $B\geq1$.
Значения `tau` лежат в $[0,1]$, включая оба конца. Contiguous не требуется:
вход может быть транспонированным срезом.

Вычисляется

$$
v_\tau^{\mathrm{VP}}=\alpha_\tau\epsilon-\sigma_\tau X_0
$$

Краевые случаи входят в контракт: $v_0^{\mathrm{VP}}=\epsilon$ и
$v_1^{\mathrm{VP}}=-X_0$, с точностью до округления $\cos\frac{\pi}{2}$.

### `vp_loss(model, x_0, eps, tau) -> Tensor`

Формы те же, что у `vp_target`: `x_0` и `eps` - `[B, 2]`, `tau` - `[B, 1]`,
$B\geq1$, значения `tau` в $[0,1]$, contiguous не требуется.

Строится $X_\tau=\alpha_\tau X_0+\sigma_\tau\epsilon$ и вычисляется

$$
\mathcal L=\frac1B\sum_{b=1}^{B}
\left\lVert \hat v_\theta^{\mathrm{VP}}(X_\tau^{(b)},\tau_b)-v_\tau^{\mathrm{VP},(b)}\right\rVert_2^2
$$

Порядок именно такой: **сначала квадрат нормы по двум координатам, потом среднее
по батчу**. Поэлементное усреднение дало бы ровно вдвое меньше.

Дополнительно:

- `model` вызывается **ровно один раз**;
- в модель подаётся $(X_\tau,\tau)$, а не $(X_0,\tau)$;
- возвращается скаляр формы `()`;
- результат дифференцируем: `.backward()` даёт градиенты параметрам модели;
- `.backward()` внутри функции не вызывается, предсказание не отрывается от
  графа.

### `euler_sample(field, z_1, nfe) -> Tensor`

`z_1` имеет форму `[B, 2]`, $B\geq1$. `field(x, tau)` принимает `[B, 2]` и
`[B, 1]`, возвращает `[B, 2]`.

Интегрирование всегда идёт от $\tau=1$ к $\tau=0$. При $N_{\mathrm{FE}}$ шагах

$$
h=-\frac{1}{N_{\mathrm{FE}}},\qquad \tau_m=1+mh,\qquad
Z^{(m+1)}=Z^{(m)}+h\,\mathrm{field}(Z^{(m)},\tau_m)
$$

то есть поле вычисляется **в начале каждого шага**: $\tau_m=1-m/N_{\mathrm{FE}}$.
Первый вызов идёт при $\tau_0=1$, значение $\tau=0$ в вызовы не попадает.

Дополнительно:

- `nfe` - положительное целое, то есть `int` или его подкласс; иначе
  `ValueError`. Исключение одно: `bool` недопустим, хотя формально наследует
  `int`. Проверка так и пишется - `isinstance(nfe, int)` с отдельным отказом
  для `bool`;
- начальный шум приходит аргументом и не меняется на месте;
- `field` вызывается **ровно `nfe` раз** - ни одного лишнего вызова для отчёта
  или диагностики;
- в каждый вызов передаётся время формы `[B, 1]`, **того же dtype и device,
  что `z_1`**;
- режим градиента выключен **в том числе во время каждого вызова `field`**, а
  не только на обновлении `z`.

## Подготовка

Ячейка ниже - общий код задания: данные, расписание, сеть, метрика, готовый
солвер RK4 и счётчик вызовов. Менять здесь ничего не нужно.

In [ ]:
import math

import torch

torch.manual_seed(20261006)
torch.set_num_threads(1)

In [ ]:
"""Инфраструктура ДЗ-2: данные, расписание, сеть, метрика, драйверы.

Всё это выдаётся студенту готовым. Его часть - три функции в отдельных
ячейках и письменные ответы.
"""

import math

import torch
from torch import nn

# --------------------------------------------------------------------------
# Данные: восемь гауссиан на окружности
# --------------------------------------------------------------------------

N_MODES = 8
RING_RADIUS = 2.0
MODE_STD = 0.15


def mode_centers(dtype=torch.float32) -> torch.Tensor:
    """Центры восьми компонент, форма [8, 2]."""
    k = torch.arange(N_MODES, dtype=dtype)
    angle = math.pi * k / 4
    return RING_RADIUS * torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)


def sample_data(n: int, generator: torch.Generator,
                dtype=torch.float32) -> torch.Tensor:
    """n точек из смеси восьми гауссиан, форма [n, 2]."""
    centers = mode_centers(dtype=dtype)
    which = torch.randint(N_MODES, (n,), generator=generator)
    noise = torch.randn(n, 2, generator=generator, dtype=dtype)
    return centers[which] + MODE_STD * noise


# --------------------------------------------------------------------------
# Расписание: нормированный угловой путь
# --------------------------------------------------------------------------

def angular_schedule(tau: torch.Tensor):
    """Возвращает (alpha_tau, sigma_tau) для углового пути.

    alpha = cos(pi*tau/2), sigma = sin(pi*tau/2). При tau=0 это чистые данные,
    при tau=1 - чистый шум.
    """
    phi = (math.pi / 2) * tau
    return torch.cos(phi), torch.sin(phi)


# --------------------------------------------------------------------------
# Сеть
# --------------------------------------------------------------------------

class VelocityNet(nn.Module):
    """Маленькая сеть: вход (x, tau) -> предсказание v^VP."""

    def __init__(self, width: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, 2),
        )

    def forward(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, tau], dim=-1))


def make_model(seed: int = 0, width: int = 64) -> VelocityNet:
    """Сеть с воспроизводимой инициализацией."""
    state = torch.get_rng_state()
    try:
        torch.manual_seed(seed)
        model = VelocityNet(width=width)
    finally:
        torch.set_rng_state(state)
    return model


def model_ode_field(model: nn.Module, x: torch.Tensor,
                    tau: torch.Tensor) -> torch.Tensor:
    """Поле ОДУ из предсказания сети: U_tau = (pi/2) * v^VP."""
    return (math.pi / 2) * model(x, tau)


# --------------------------------------------------------------------------
# Метрика: energy distance
# --------------------------------------------------------------------------

def energy_distance(x: torch.Tensor, y: torch.Tensor) -> float:
    """Оценка energy distance между двумя выборками. Считается в float64."""
    x = x.detach().to(torch.float64)
    y = y.detach().to(torch.float64)
    n, m = x.shape[0], y.shape[0]
    cross = torch.cdist(x, y).sum() / (n * m)
    inner_x = torch.cdist(x, x).sum() / (n * n)
    inner_y = torch.cdist(y, y).sum() / (m * m)
    return float(2 * cross - inner_x - inner_y)


# --------------------------------------------------------------------------
# Солверы и счётчик вызовов
# --------------------------------------------------------------------------

class CountingField:
    """Обёртка над полем, считающая вызовы и запоминающая переданные времена."""

    def __init__(self, field):
        self.field = field
        self.calls = 0
        self.taus: list[float] = []

    def __call__(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        self.taus.append(float(tau.reshape(-1)[0]))
        return self.field(x, tau)


def rk4_sample(field, z_1: torch.Tensor, nfe: int) -> torch.Tensor:
    """Классический RK4 с фиксированным шагом. Один шаг - четыре вызова поля.

    Дан готовым: реализовывать его не нужно, он служит второй кривой в
    сравнении при одинаковом бюджете вызовов.
    """
    if not isinstance(nfe, int) or isinstance(nfe, bool) or nfe <= 0:
        raise ValueError("nfe должно быть положительным int")
    if nfe % 4:
        raise ValueError("для RK4 бюджет вызовов должен делиться на 4")

    steps = nfe // 4
    h = -1.0 / steps
    z = z_1.clone()
    ones = torch.ones(z.shape[0], 1, dtype=z.dtype, device=z.device)

    with torch.no_grad():
        for m in range(steps):
            tau = 1.0 + m * h
            k1 = field(z, ones * tau)
            k2 = field(z + 0.5 * h * k1, ones * (tau + 0.5 * h))
            k3 = field(z + 0.5 * h * k2, ones * (tau + 0.5 * h))
            k4 = field(z + h * k3, ones * (tau + h))
            z = z + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    return z

## Разобранный пример: условный таргет

Возьмём

$$
x_0=(2,0),\qquad \epsilon=(0,1),\qquad \tau=\tfrac12
$$

Тогда $\alpha_{1/2}=\sigma_{1/2}=\tfrac1{\sqrt2}$, и зашумлённая точка равна

$$
x_{1/2}=\tfrac1{\sqrt2}(x_0+\epsilon)=\left(\sqrt2,\tfrac1{\sqrt2}\right)
$$

Условный таргет:

$$
v_{1/2}^{\mathrm{VP}}=\tfrac1{\sqrt2}(\epsilon-x_0)
=\left(-\sqrt2,\tfrac1{\sqrt2}\right)
$$

**Важно понимать, чего сеть при этом не делает.** Она не обязана возвращать
именно этот вектор в точке $x_{1/2}$. Ту же точку могли дать другие пары
$(x_0,\epsilon)$ с другими таргетами. В оптимуме сеть возвращает среднее по всем
таким парам - это и есть marginal-объект. Почему так - первый письменный вопрос.

In [ ]:
ex_x0 = torch.tensor([[2.0, 0.0]], dtype=torch.float64)
ex_eps = torch.tensor([[0.0, 1.0]], dtype=torch.float64)
ex_tau = torch.tensor([[0.5]], dtype=torch.float64)

alpha, sigma = angular_schedule(ex_tau)
print("alpha, sigma:", float(alpha), float(sigma))
print("x_tau посчитано:", (alpha * ex_x0 + sigma * ex_eps).flatten().tolist())
print("x_tau ожидалось:", [math.sqrt(2.0), 1 / math.sqrt(2.0)])
print("таргет ожидался:", [-math.sqrt(2.0), 1 / math.sqrt(2.0)])

## Задание 1. Условный таргет и функция потерь

Заполните обе заготовки. Всё, что нужно, есть в контракте выше.

In [ ]:
def vp_target(x_0: torch.Tensor, eps: torch.Tensor,
              tau: torch.Tensor) -> torch.Tensor:
    """Условный таргет v^VP. Формы: [B,2], [B,2], [B,1] -> [B,2]."""
    raise NotImplementedError


def vp_loss(model, x_0: torch.Tensor, eps: torch.Tensor,
            tau: torch.Tensor) -> torch.Tensor:
    """Средний квадрат нормы ошибки предсказания v^VP. Возвращает скаляр."""
    raise NotImplementedError

## От условного таргета к полю для генерации

Сеть обучается предсказывать $v^{\mathrm{VP}}$. Для интегрирования нужно поле

$$
v_\theta^{\mathrm{ODE}}(x,\tau)=\frac{\pi}{2}\hat v_\theta^{\mathrm{VP}}(x,\tau)
$$

Функция `model_ode_field` уже делает это умножение. Ваш sampler получает поле
аргументом и про сеть ничего не знает - поэтому его можно проверять на
искусственных полях, где ответ известен точно.

### Разобранный пример: постоянное поле

Пусть $\mathrm{field}(x,\tau)=c$. Интегрирование от $\tau=1$ до $\tau=0$ даёт

$$
Z_0=Z_1-c
$$

Явный Эйлер обязан получить это **точно при любом числе шагов**: сумма
$N_{\mathrm{FE}}$ шагов длины $h=-1/N_{\mathrm{FE}}$ равна $-1$. Это же
объясняет, откуда берётся минус: его даёт шаг, а не отдельный знак перед полем.

## Задание 2. Sampler

Заполните заготовку по контракту выше.

In [ ]:
def euler_sample(field, z_1: torch.Tensor, nfe: int) -> torch.Tensor:
    """Явный Эйлер от tau=1 к tau=0 за nfe шагов. Форма как у z_1."""
    raise NotImplementedError

## Самопроверка

Здесь несколько базовых проверок. Полный комплект шире: он включает сверку с
независимым эталоном, аналитические градиенты, обе точности и краевые размеры
батча.

Соответствие духу задания - честность реализации, отсутствие подгонки под эти
проверки - оценивается чтением кода, см. «Как оценивается».

**Всё, что ниже этого заголовка, при проверке не исполняется.** Оцениваемый код
должен быть выше.

In [ ]:
def _close(got, want, rtol=1e-10, atol=1e-12):
    return torch.allclose(got, want, rtol=rtol, atol=atol)


def _try(thunk):
    """Возвращает True, False или None, если функция ещё не реализована."""
    try:
        return bool(thunk())
    except NotImplementedError:
        return None
    except Exception:
        return False


def selfcheck():
    cases = []
    g = torch.Generator().manual_seed(0)
    x_0 = torch.randn(6, 2, generator=g, dtype=torch.float64)
    eps = torch.randn(6, 2, generator=g, dtype=torch.float64)
    zeros = torch.zeros(6, 1, dtype=torch.float64)
    ones = torch.ones(6, 1, dtype=torch.float64)

    cases.append(("tau=0: таргет равен eps",
                  _try(lambda: _close(vp_target(x_0, eps, zeros), eps))))
    cases.append(("tau=1: таргет равен -x_0",
                  _try(lambda: _close(vp_target(x_0, eps, ones), -x_0, atol=1e-15))))

    ex_x0 = torch.tensor([[2.0, 0.0]], dtype=torch.float64)
    ex_eps = torch.tensor([[0.0, 1.0]], dtype=torch.float64)
    ex_tau = torch.tensor([[0.5]], dtype=torch.float64)
    ex_want = torch.tensor([[-math.sqrt(2.0), 1 / math.sqrt(2.0)]],
                           dtype=torch.float64)
    cases.append(("разобранный пример при tau=0.5",
                  _try(lambda: _close(vp_target(ex_x0, ex_eps, ex_tau), ex_want))))

    class _Zero(torch.nn.Module):
        def forward(self, x, tau):
            return torch.zeros_like(x)

    tau = torch.rand(6, 1, generator=g, dtype=torch.float64)
    cases.append(("нулевая модель: потери равны норме таргета",
                  _try(lambda: _close(vp_loss(_Zero(), x_0, eps, tau),
                                      (vp_target(x_0, eps, tau) ** 2).sum(dim=-1).mean()))))

    def _grad_flows():
        model = make_model(seed=7)
        loss = vp_loss(model, x_0.float(), eps.float(), tau.float())
        loss.backward()
        grads = [p.grad for p in model.parameters() if p.grad is not None]
        return bool(grads) and any(float(g.abs().max()) > 0 for g in grads)

    cases.append(("после backward есть ненулевые градиенты", _try(_grad_flows)))

    c = torch.tensor([[0.3, -0.7]], dtype=torch.float64)
    z_1 = torch.randn(4, 2, generator=g, dtype=torch.float64)
    cases.append(("постоянное поле: z_0 = z_1 - c при nfe=1",
                  _try(lambda: _close(euler_sample(lambda x, t: c.expand_as(x), z_1, 1),
                                      z_1 - c))))
    cases.append(("постоянное поле: то же самое при nfe=16",
                  _try(lambda: _close(euler_sample(lambda x, t: c.expand_as(x), z_1, 16),
                                      z_1 - c))))

    def _counts():
        counter = CountingField(lambda x, t: torch.zeros_like(x))
        euler_sample(counter, z_1, 5)
        return counter.calls == 5

    cases.append(("поле вызвано ровно nfe раз", _try(_counts)))

    mark = {True: "OK     ", False: "FAIL   ", None: "не готов"}
    for name, ok in cases:
        print(f"{mark[ok]}  {name}")
    done = sum(1 for _, o in cases if o is True)
    todo = sum(1 for _, o in cases if o is None)
    print(f"\nпройдено {done} из {len(cases)}"
          + (f", ещё не реализовано: {todo}" if todo else ""))


selfcheck()

## Обучение

Драйвер дан готовым: данные, оптимизатор, число шагов и seed'ы задаются
условием, чтобы прогон был воспроизводим в одной среде и сопоставим между
работами. Побитового совпадения весов между разными машинами ждать не стоит.
На CPU обучение занимает меньше минуты.

Обучение нужно, чтобы увидеть, что ваш loss работает. Для обязательных
измерений ниже используется **общий замороженный checkpoint**: иначе кривая
качества у разных людей отражала бы разное обучение, а не разные солверы.

In [ ]:
"""Обучающий драйвер ДЗ-2. Выдаётся студенту готовым."""

import torch


TRAIN_CONFIG = {
    "batch_size": 512,
    "steps": 12000,
    "lr": 2e-3,
    "data_seed": 1234,
    "tau_seed": 5678,
    "model_seed": 0,
    "report_every": 2000,
}


def train_model(loss_fn, cfg=None, verbose: bool = True):
    """Обучает сеть по переданной функции потерь. Возвращает (model, история).

    `loss_fn(model, x_0, eps, tau)` - ваша реализация. Всё остальное задано:
    данные, расписание, оптимизатор, seed'ы. Менять здесь ничего не нужно.
    """
    cfg = dict(TRAIN_CONFIG if cfg is None else cfg)
    model = make_model(seed=cfg["model_seed"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    data_gen = torch.Generator().manual_seed(cfg["data_seed"])
    tau_gen = torch.Generator().manual_seed(cfg["tau_seed"])
    batch = cfg["batch_size"]
    history = []

    for step in range(1, cfg["steps"] + 1):
        x_0 = sample_data(batch, data_gen)
        eps = torch.randn(batch, 2, generator=tau_gen)
        tau = torch.rand(batch, 1, generator=tau_gen)

        loss = loss_fn(model, x_0, eps, tau)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.detach().item())

        if verbose and step % cfg["report_every"] == 0:
            window = history[-cfg["report_every"]:]
            print(f"  шаг {step:6d}   потери {sum(window) / len(window):.4f}")

    return model, history

In [ ]:
try:
    model, history = train_model(vp_loss)
    print("\nсредние потери за последние 500 шагов:",
          round(sum(history[-500:]) / 500, 4))
except NotImplementedError:
    print("vp_loss ещё не реализована - вернитесь сюда, когда будет готова")

## Измерения

Протокол задан полностью и менять его не нужно.

- сетка $N_{\mathrm{FE}}$: 4, 8, 16, 32, 64;
- две кривые: ваш Эйлер и готовый RK4 при **одинаковом бюджете вызовов**;
  у RK4 один шаг солвера стоит четыре вызова, поэтому при $N_{\mathrm{FE}}=4$
  у него всего один шаг;
- 2048 сгенерированных точек, 5 повторов с зафиксированными seed'ами;
- **один и тот же начальный шум** используется для всех $N_{\mathrm{FE}}$ и
  обоих солверов: сравнение парное;
- эталонная выборка из данных тоже фиксирована;
- в файл пишется **каждый повтор отдельной строкой**, а не только среднее.
  Это важно: по средним парность уже не восстановить, а именно она отвечает на
  вопрос про солверы.

### Метрика

Energy distance между сгенерированной выборкой и выборкой из данных:

$$
D_E(P,Q)=2\,\mathbb E\lVert X-Y\rVert-\mathbb E\lVert X-X'\rVert-\mathbb E\lVert Y-Y'\rVert
$$

Реализация дана готовой и считается в float64.

Чего она не делает: это не абсолютная оценка качества. Величина зависит от
масштаба пространства, и разные дефекты она сворачивает в одно число.

Рядом печатается **ориентир конечновыборочного шума**: energy distance между
двумя независимыми выборками из самих данных, посчитанный по двадцати парам, со
средним и разбросом.

Это **не** нижняя граница. Величина случайна: другая пара выборок из тех же
данных вполне может дать меньше. Ориентир отвечает на вопрос «какого порядка
здесь шум оценки», а не «ниже чего нельзя опуститься».

In [ ]:
"""Драйвер измерений ДЗ-2. Выдаётся студенту готовым."""

import math

import torch


MEASURE_CONFIG = {
    "nfe_grid": (4, 8, 16, 32, 64),
    "n_samples": 2048,
    "noise_seeds": (11, 22, 33, 44, 55),
    "data_seed": 99,
    "baseline_pairs": 20,
}

COLUMNS = ("solver,nfe,solver_steps,repeat,noise_seed,n_samples,"
           "energy_distance").split(",")


def reference_batch(cfg=MEASURE_CONFIG) -> torch.Tensor:
    """Фиксированная выборка из данных, одна и та же для всей таблицы."""
    return sample_data(cfg["n_samples"],
                       torch.Generator().manual_seed(cfg["data_seed"]))


def noise_batches(cfg=MEASURE_CONFIG) -> list:
    """Наборы начального шума с зафиксированными seed'ами.

    Один и тот же шум используется для всех NFE и обоих солверов: сравнение
    парное, и разброс между повторами не смешивается с разбросом между
    настройками.
    """
    return [torch.randn(cfg["n_samples"], 2,
                        generator=torch.Generator().manual_seed(seed))
            for seed in cfg["noise_seeds"]]


def mean_std(values):
    """Среднее и выборочное стандартное отклонение, ddof = 1."""
    n = len(values)
    mean = sum(values) / n
    var = sum((v - mean) ** 2 for v in values) / (n - 1) if n > 1 else 0.0
    return mean, math.sqrt(var)


def data_baseline(cfg=MEASURE_CONFIG):
    """Energy distance между двумя независимыми выборками из данных.

    Это **не** нижняя граница: величина случайна, и другая пара выборок вполне
    может дать меньше. Считается по многим парам и возвращается со средним и
    разбросом - как ориентир конечновыборочного шума, а не как порог.
    """
    reference = reference_batch(cfg)
    values = []
    for i in range(cfg["baseline_pairs"]):
        other = sample_data(cfg["n_samples"],
                            torch.Generator().manual_seed(1000 + i))
        values.append(energy_distance(reference, other))
    mean, std = mean_std(values)
    return {"mean": mean, "std": std, "min": min(values), "max": max(values),
            "pairs": cfg["baseline_pairs"]}


def run_measurements(model, euler_sample, path="measurements.csv",
                     cfg=MEASURE_CONFIG, verbose=True):
    """Кривая качества по числу вызовов сети для Эйлера и готового RK4.

    В файл пишется каждый повтор отдельной строкой, а не только среднее:
    эксперимент парный, и по агрегату парность уже не восстановить.
    Менять здесь ничего не нужно.
    """
    torch.set_num_threads(1)
    reference = reference_batch(cfg)
    noises = noise_batches(cfg)
    field = lambda x, tau: model_ode_field(model, x, tau)
    rows = []

    for solver_name, sampler, per_step in (("euler", euler_sample, 1),
                                           ("rk4", rk4_sample, 4)):
        for nfe in cfg["nfe_grid"]:
            for repeat, (seed, z_1) in enumerate(zip(cfg["noise_seeds"], noises), 1):
                # копия на каждый вызов: солвер с ошибкой на месте иначе
                # испортил бы все последующие точки, включая готовый RK4
                score = energy_distance(sampler(field, z_1.clone(), nfe),
                                        reference)
                rows.append([solver_name, nfe, nfe // per_step, repeat, seed,
                             cfg["n_samples"], repr(score)])

    with open(path, "w", encoding="utf-8") as f:
        f.write(",".join(COLUMNS) + "\n")
        for row in rows:
            f.write(",".join(str(x) for x in row) + "\n")

    if verbose:
        summarize(rows, cfg)
        print(f"\nзаписано строк: {len(rows)} -> {path}")
    return rows


def _by_setting(rows):
    grouped = {}
    for solver, nfe, steps, repeat, seed, n, score in rows:
        grouped.setdefault((solver, int(nfe), int(steps)), []).append(float(score))
    return grouped


def summarize(rows, cfg=MEASURE_CONFIG):
    """Печатает сводку и парные разности между солверами."""
    grouped = _by_setting(rows)
    print(f"  {'солвер':<8} {'NFE':>4} {'шагов':>6} {'среднее':>10} {'разброс':>10}")
    for (solver, nfe, steps), values in sorted(grouped.items()):
        mean, std = mean_std(values)
        print(f"  {solver:<8} {nfe:>4} {steps:>6} {mean:>10.5f} {std:>10.5f}")

    _paired(grouped, cfg)


def _paired(grouped, cfg):
    """Парные разности: между солверами и между соседними NFE.

    Начальный шум у всех точек один и тот же, поэтому парной является не
    только пара солверов, но и пара соседних бюджетов. По средним с разбросом
    вывод о плато можно сделать неверный: разброс между повторами обычно
    больше самой разности.
    """
    print("\n  парная разность Эйлер минус RK4 при равном числе вызовов")
    print(f"  {'NFE':>5} {'среднее':>11} {'разброс':>11}  знак одинаков")
    for nfe in cfg["nfe_grid"]:
        euler = grouped.get(("euler", nfe, nfe))
        rk4 = grouped.get(("rk4", nfe, nfe // 4))
        if not euler or not rk4:
            continue
        diffs = [a - b for a, b in zip(euler, rk4)]
        mean, std = mean_std(diffs)
        same = "да" if all(d > 0 for d in diffs) or all(d < 0 for d in diffs) else "нет"
        print(f"  {nfe:>5} {mean:>11.5f} {std:>11.5f}  {same:>13}")

    for solver in ("euler", "rk4"):
        print(f"\n  парная разность между соседними NFE, {solver}")
        print(f"  {'переход':>12} {'среднее':>11} {'разброс':>11}  знак одинаков")
        grid = cfg["nfe_grid"]
        for low, high in zip(grid, grid[1:]):
            steps = (lambda n: n if solver == "euler" else n // 4)
            a = grouped.get((solver, low, steps(low)))
            b = grouped.get((solver, high, steps(high)))
            if not a or not b:
                continue
            diffs = [x - y for x, y in zip(a, b)]
            mean, std = mean_std(diffs)
            same = ("да" if all(d > 0 for d in diffs) or all(d < 0 for d in diffs)
                    else "нет")
            print(f"  {f'{low} -> {high}':>12} {mean:>11.5f} {std:>11.5f}"
                  f"  {same:>13}")

In [ ]:
frozen = make_model(seed=0)
frozen.load_state_dict(torch.load("checkpoint.pt", weights_only=True))
frozen.eval()

baseline = data_baseline()
print(f"данные против данных, {baseline['pairs']} пар: "
      f"среднее {baseline['mean']:.5f}, разброс {baseline['std']:.5f}, "
      f"от {baseline['min']:.5f} до {baseline['max']:.5f}\n")

try:
    rows = run_measurements(frozen, euler_sample)
except NotImplementedError:
    print("euler_sample ещё не реализована - вернитесь сюда, когда будет готова")

## Анализ измерений

Опишите, что показали **ваши** числа. Обязательно:

- где изменение $N_{\mathrm{FE}}$ ещё даёт заметный выигрыш, а где перестаёт;
- как величина оставшегося зазора соотносится с ориентиром шума метрики, и что
  из этого следует про то, можно ли вообще по этой таблице отделить ошибку поля
  от шума оценки;
- что получилось при сравнении Эйлера и RK4 **при одинаковом бюджете вызовов**;
- **сравните два взгляда на одни и те же числа**: разброс внутри каждого солвера
  и парную разность по повторам. Они могут говорить разное - объясните, почему,
  и какой из них отвечает на вопрос «какой солвер лучше при этом бюджете»;
- почему из этой таблицы нельзя сделать вывод «RK4 вообще лучше» или «вообще
  хуже».

Конкретных чисел не требуется: засчитывается честное описание своих данных с
объяснением.

*Ваш ответ:*

## Письменные ответы

### 1. Почему регрессия на условный таргет обучает marginal-объект

Обязательно пройти четыре шага:

1. для квадратичной ошибки при фиксированных $X_\tau=x$ и $\tau$ оптимум - это
   $\mathbb E[v_\tau^{\mathrm{VP}}\mid X_\tau=x,\tau]$;
2. для углового пути $U_\tau=\frac{\pi}{2}v_\tau^{\mathrm{VP}}$;
3. отсюда $\mathbb E[U_\tau\mid X_\tau=x,\tau]=\frac{\pi}{2}\mathbb E[v_\tau^{\mathrm{VP}}\mid X_\tau=x,\tau]$;
4. результат о маргинализации из flow matching говорит, что
   $v_\tau(x)=\mathbb E[U_\tau\mid X_\tau=x]$ порождает нужный probability path.

Отдельно скажите: шаг 1 - утверждение про регрессию, а не про перенос массы.
Утверждение про перенос даёт шаг 4, и без него цепочка не замыкается. И отдельно
различите идеальный предсказатель в популяционном оптимуме и реально обученную
сеть конечной ёмкости.

*Ваш ответ:*

### 2. Три объекта, которые обозначают буквой $v$

Дайте определение $v_\tau^{\mathrm{VP}}$, $U_\tau$ и $v_\tau(x)$; скажите, какие
из них зависят от скрытой пары $(X_0,\epsilon)$, а какой является однозначной
функцией точки и времени; выведите $U_\tau=\frac{\pi}{2}v_\tau^{\mathrm{VP}}$
для углового пути; объясните, почему совпадение буквы в литературе не делает эти
величины одним объектом.

*Ваш ответ:*

### 3. Направление и стоимость шага

Почему генерация идёт от $\tau=1$ к $\tau=0$ и где в коде это выражено. Чем шаг
солвера отличается от вызова сети, сколько вызовов стоит один шаг у Эйлера и у
RK4, и почему сравнивать солверы надо при равном числе вызовов, а не при равном
числе шагов.

*Ваш ответ:*

## Описание применения генеративных моделей

Обязательный раздел, 150-250 слов. Выберите одно применение и укажите явно:

- что является объектом $x$;
- что означает целевое распределение;
- что значит получить новую выборку;
- зачем здесь нужна именно генеративная модель, а не предсказывающая;
- один практически значимый способ, которым такая модель может подвести.

Дополнительных экспериментов и картинок для этого раздела не требуется.

*Ваш ответ:*

## Как оценивается

| Критерий | Вес | Что оценивается |
|---|---|---|
| Корректность | 20 % | формулы и контракт, работа воспроизводится в чистом ядре |
| Письменный вывод | 30 % | три ответа и описание применения, каждый переход обоснован |
| Измерения | 25 % | замеры воспроизводятся одним прогоном, условия зафиксированы, сделаны повторы |
| Анализ и выводы | 25 % | плато и порядок кривых объяснены, разброс между повторами учтён, ограничения названы |

Двадцать процентов за автотесты - сознательно небольшая доля. Основной вес
приходится на вывод, качество измерений и анализ.

Автотесты проверяют то, что детерминировано: значение таргета, значение и
градиенты потерь на заданной модели, шаг солвера, сетку времён и точное число
вызовов поля. **Результат обучения автотестами не проверяется** - он зависит от
версии библиотек и не является контрактом.

**Что входит в корректность помимо автотестов.** Тесты видят формулы, формы,
точности, градиенты, отсутствие скрытой случайности, число вызовов поля и
отключённый граф при выборке. Чтением кода проверяется то, чего они не видят:
решение написано для произвольного входа, а не подогнано под открытые
самопроверки; выданная инфраструктура не изменена - расписание, датасет, сеть,
драйверы обучения и измерений. Дважды за одну ошибку не наказывают: то, что уже
отвергнуто тестом, второй раз не снимается.

## Как сдавать

Сдаётся этот же ноутбук, заполненный, с сохранёнными результатами выполнения
ячеек, и файл `measurements.csv`.

Перед сдачей выполните ноутбук сверху вниз в чистом ядре и убедитесь, что он
проходит целиком.

Оценивается код **выше раздела «Самопроверка»**. Магии IPython (`%`, `%%`) в
оцениваемой части быть не должно. На прогон комплекта на вашей работе отводится
120 секунд.

Просрочка: до 24 часов −20 %, от 24 до 48 часов −40 %, от 48 до 72 часов −60 %.
Позже 72 часов работа не принимается. Отсчёт от 23:59 13.10 по времени Москвы.